# 2024 — Multi-Token Prediction (MTP)

**Papers:** Gloeckle et al., *"Better & Faster Large Language Models via Multi-token Prediction"* (Meta, 2024); DeepSeek-AI, *"DeepSeek-V3 Technical Report"* (2024), whose sequential MTP design is kept in DeepSeek-V4 (2026, MTP depth 1, loss weight 0.3 → 0.1); Qwen3-Next and Qwen3.5 also train with MTP.

Next-token prediction gives one learning signal per position. **MTP** adds small extra modules that predict tokens **further ahead**. In DeepSeek-V3's sequential design, MTP module $k$ combines the previous depth's hidden state with the embedding of the token $k$ steps ahead and predicts the token after it:

$$h'^{\,k}_i = M_k\big[\operatorname{RMSNorm}(h^{k-1}_i);\ \operatorname{RMSNorm}(\operatorname{Emb}(t_{i+k}))\big],\qquad h^k_i = \operatorname{TRM}_k(h'^{\,k}_i),\qquad P^k_{i+k+1} = \operatorname{Head}(h^k_i)$$

The embedding and output head are **shared** with the main model, and the total loss is $\mathcal L_{main} + \lambda\,\mathcal L_{MTP}$. Two benefits:
1. **Denser training signal:** the model must plan representations that are useful several tokens ahead.
2. **Free drafting at inference:** the MTP module guesses the token after next, so it can serve as the draft model for speculative decoding (notebook 60) without a separate model. DeepSeek reports 85–90% acceptance for the second token.

In [1]:
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def get_device():
    """CUDA > Apple MPS > CPU. Force one with the env var NN_DEVICE=cpu|cuda|mps."""
    forced = os.environ.get("NN_DEVICE")
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = get_device()
torch.set_flush_denormal(True)   # tiny "denormal" floats are extremely slow on CPUs; treat them as 0
seed_everything(0)
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.15.0 | device: mps


In [2]:
# A procedurally generated corpus (~290K characters) with a real held-out split.
_rng = random.Random(0)
_subjects = ["the model", "a small network", "the optimizer", "each layer", "the attention head", "the router", "a student", "the teacher"]
_verbs = ["learns", "predicts", "updates", "compresses", "routes", "remembers", "forgets", "copies"]
_objects = ["the next token", "a hidden state", "the gradient", "its weights", "a long context", "the key and value", "every expert", "the loss"]
_advs = ["quickly", "slowly", "carefully", "in parallel", "step by step", "without labels", "at scale", "once again"]
def _sentence():
    s = f"{_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)} {_rng.choice(_advs)}"
    if _rng.random() < 0.4: s += f" because {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    if _rng.random() < 0.2: s += f", then {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    return s[0].upper() + s[1:] + ". "
text = "".join(_sentence() for _ in range(4000))
chars = sorted(set(text)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
split = int(0.9 * len(data)); train_data, val_data = data[:split], data[split:]
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(chars[i] for i in ids)
print(f"{len(text):,} characters, vocab {V}")

290,321 characters, vocab 31


In [3]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj, self.h = nn.Linear(d, 3 * d), nn.Linear(d, d), heads
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        x = x + self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    """A small GPT-2-style character model (pre-norm blocks, learned positions)."""
    def __init__(self, V, d=128, n_layers=4, heads=4, ctx=128):
        super().__init__()
        self.ctx = ctx
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V, bias=False)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks: x = b(x)
        return self.head(self.ln(x))

def train_gpt(model, train_data, steps=800, B=32, lr=2e-3):
    model = model.to(train_data.device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.1)
    for s in range(steps):
        ix = torch.randint(0, len(train_data) - model.ctx - 1, (B,), device=train_data.device)
        x = train_data[ix[:, None] + torch.arange(model.ctx, device=train_data.device)]; y = train_data[ix[:, None] + torch.arange(1, model.ctx + 1, device=train_data.device)]
        loss = F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def eval_loss(model, data, n=20, B=32):
    g = torch.Generator(device="cpu").manual_seed(123); losses = []
    for _ in range(n):
        ix = torch.randint(0, len(data) - model.ctx - 1, (B,), generator=g).to(data.device)
        x = data[ix[:, None] + torch.arange(model.ctx, device=data.device)]; y = data[ix[:, None] + torch.arange(1, model.ctx + 1, device=data.device)]
        losses.append(F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1)).item())
    return float(np.mean(losses))

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6): super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x): return self.w * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

class GPTWithMTP(GPT):
    """GPT + one DeepSeek-V3-style MTP module (depth 1) that predicts token i+2 from h_i and Emb(t_{i+1})."""
    def __init__(self, V, d=128, n_layers=4, heads=4, ctx=128):
        super().__init__(V, d, n_layers, heads, ctx)
        self.mtp_norm_h, self.mtp_norm_e = RMSNorm(d), RMSNorm(d)
        self.mtp_proj = nn.Linear(2 * d, d, bias=False)                                  # M_1
        self.mtp_block = Block(d, heads)                                                 # TRM_1 (causal)
    def hidden(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks: x = b(x)
        return x
    def forward_mtp(self, idx):
        h = self.hidden(idx)                                                             # (B, T, d)
        main_logits = self.head(self.ln(h))                                              # predicts t_{i+1}
        e_next = self.tok(idx[:, 1:])                                                    # Emb(t_{i+1}): shared embedding
        h1 = self.mtp_proj(torch.cat([self.mtp_norm_h(h[:, :-1]), self.mtp_norm_e(e_next)], -1))
        h1 = self.mtp_block(h1 + self.pos(torch.arange(h1.shape[1], device=idx.device)))
        mtp_logits = self.head(self.ln(h1))                                              # predicts t_{i+2}; shared head
        return main_logits, mtp_logits

def train(use_mtp, steps=800, lam=0.3, B=32):
    seed_everything(0)
    m = (GPTWithMTP(V) if use_mtp else GPT(V)).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-3, weight_decay=0.1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps, pct_start=0.1)
    for s in range(steps):
        ix = torch.randint(0, len(train_data) - m.ctx - 2, (B,), device=device)
        x = train_data[ix[:, None] + torch.arange(m.ctx, device=train_data.device)]
        y1 = train_data[ix[:, None] + torch.arange(1, m.ctx + 1, device=train_data.device)]
        if use_mtp:
            main, mtp = m.forward_mtp(x)
            y2 = train_data[ix[:, None] + torch.arange(2, m.ctx + 1, device=train_data.device)]            # targets two steps ahead
            loss = F.cross_entropy(main.reshape(-1, V), y1.reshape(-1)) + lam * F.cross_entropy(mtp.reshape(-1, V), y2.reshape(-1))
        else:
            loss = F.cross_entropy(m(x).reshape(-1, V), y1.reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step()
    return m.eval()

t0 = time.time()
base, mtp_model = train(False), train(True)
print(f"trained both in {time.time() - t0:.0f}s")
print(f"next-token validation loss | plain GPT: {eval_loss(base, val_data):.4f} | GPT trained with MTP: {eval_loss(mtp_model, val_data):.4f}")
print("(the MTP module adds parameters only for training and drafting; the main next-token path is the same architecture)")
print("At this toy scale both models reach the same loss floor of the corpus; the paper-scale gain from the denser signal does not show here.")

trained both in 139s


next-token validation loss | plain GPT: 0.2143 | GPT trained with MTP: 0.2143
(the MTP module adds parameters only for training and drafting; the main next-token path is the same architecture)
At this toy scale both models reach the same loss floor of the corpus; the paper-scale gain from the denser signal does not show here.


## Using the MTP module as a free draft model
At each position the main head predicts $t_{i+1}$; feeding that guess back into the MTP module predicts $t_{i+2}$. How often does that guess match what the main model itself would predict one step later (greedy)? That is the acceptance rate for self-speculative decoding.

In [4]:
@torch.no_grad()
def mtp_acceptance(m, n_batches=10, B=16):
    """At position i the main head guesses g1 = t_{i+1} and the MTP module guesses g2 = t_{i+2}.
    Verifying g2 means running the main model on prefix + g1. Wherever g1 equals the real next token,
    that is just the main model's own prediction at position i+1 on the real text, so we can check
    every such position in one batched pass."""
    g = torch.Generator().manual_seed(1); agree = total = 0
    for _ in range(n_batches):
        ix = torch.randint(0, len(val_data) - m.ctx - 2, (B,), generator=g).to(device)
        x = val_data[ix[:, None] + torch.arange(m.ctx, device=val_data.device)]
        h = m.hidden(x); g1 = m.head(m.ln(h)).argmax(-1)                                  # main model's greedy guesses
        h1 = m.mtp_proj(torch.cat([m.mtp_norm_h(h), m.mtp_norm_e(m.tok(g1))], -1))
        g2 = m.head(m.ln(m.mtp_block(h1 + m.pos(torch.arange(h1.shape[1], device=device))))).argmax(-1)
        ok = g1[:, :-1] == x[:, 1:]                                                      # positions where the prefix+g1 is the real text
        agree += ((g2[:, :-1] == g1[:, 1:]) & ok).sum().item(); total += ok.sum().item()
    return agree / total
rate = mtp_acceptance(mtp_model)
print(f"MTP draft of the 2nd token agrees with the main model's own greedy choice {rate:.0%} of the time")
print(f"=> self-speculative decoding would emit about {1 + rate:.2f} tokens per forward pass (greedy)")

MTP draft of the 2nd token agrees with the main model's own greedy choice 93% of the time
=> self-speculative decoding would emit about 1.93 tokens per forward pass (greedy)
